In [6]:
"""
    legendre_polinomio(n, x)

Evalúa el n-ésimo Polinomio de Legendre P_n(x) en el punto x usando la recurrencia.
"""
function legendre_polinomio(n::Int, x::Real)
    if n == 0
        return 1.0
    elseif n == 1
        return x
    else
        P_prev2 = 1.0  # P_0
        P_prev1 = x    # P_1
        for k = 2:n
            # Relación de recurrencia de Bonnet
            P_k = ((2k - 1) * x * P_prev1 - (k - 1) * P_prev2) / k
            P_prev2 = P_prev1
            P_prev1 = P_k
        end
        return P_prev1
    end
end

"""
    aproximacion_legendre(f, grado, N_puntos)

Calcula la aproximación de Legendre del grado dado para f(x) en [-1, 1].
Usa la Regla del Punto Medio Compuesta para la integración numérica.

# devuelve
- un vector de coeficientes Legendre (c_0, c_1, ..., c_m).
"""
function aproximacion_legendre(f::Function, grado::Int, N_puntos::Int = 1000)
    coeficientes = zeros(grado + 1)
    a, b = -1.0, 1.0
    h = (b - a) / N_puntos # ancho de cada subintervalo

    for k = 0:grado
        integral_sum = 0.0
        
        # integración numérica usando la Regla del Punto Medio Compuesta
        for i = 1:N_puntos
            # xi es el punto medio del subintervalo [a + (i-1)h, a + i*h]
            xi = a + (i - 0.5) * h 
            
            P_k_xi = legendre_polinomio(k, xi)
            # acumulamos f(xi) * P_k(xi)
            integral_sum += f(xi) * P_k_xi
        end
        
        integral_estimada = integral_sum * h # multiplicamos por el ancho de cada subintervalo h
        
        # fórmula para el coeficiente c_k
        coeficientes[k+1] = (2k + 1) / 2.0 * integral_estimada
    end

    return coeficientes
end

"""
    evaluar_legendre_aprox(x_val, coeficientes)

Evalúa la aproximación de Legendre en un punto x_val dados los coeficientes.
"""
function evaluar_legendre_aprox(x_val::Real, coeficientes::AbstractVector{<:Real})
    y_aprox = 0.0
    for (k, c_k) in enumerate(coeficientes)
        # k-1 porque c_k[1] corresponde a P_0
        y_aprox += c_k * legendre_polinomio(k - 1, x_val)
    end
    return y_aprox
end

evaluar_legendre_aprox

In [7]:
#ejemplo
f_ejemplo(x) = cos(x)
grado_legendre = 4 # entre más alto el grado, mejor la aproximación
N_integracion = 1000 # número de puntos para la integración numérica

coefs_legendre = aproximacion_legendre(f_ejemplo, grado_legendre, N_integracion)

println("--- Aproximación Polinómica de Legendre (Grado $grado_legendre) ---")
println("Coeficientes Legendre (c_0, c_1, c_2, c_3, c_4):")
println(coefs_legendre)

# prueba
x_prueba = 0.5
valor_real = f_ejemplo(x_prueba)
valor_aprox = evaluar_legendre_aprox(x_prueba, coefs_legendre)

println("\nEvaluación en x = $x_prueba:")
println("Valor real de cos(0.5): ", valor_real)
println("Valor aproximado: ", valor_aprox)
println("Error absoluto: ", abs(valor_real - valor_aprox))

--- Aproximación Polinómica de Legendre (Grado 4) ---
Coeficientes Legendre (c_0, c_1, c_2, c_3, c_4):
[0.8414711250530765, -2.0317081350640364e-17, -0.31017590958794017, 3.963496197911809e-17, 0.009092299934971963]

Evaluación en x = 0.5:
Valor real de cos(0.5): 0.8775825618903728
Valor aproximado: 0.8776148708016163
Error absoluto: 3.230891124350599e-5
